In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Имеется информация за $10$ дней в агрегаторе такси:

- Количество заказов = $8770$.
- Среднее время поиска и назначения водителя на заказ составляет $2$ минуты и $1$ секунду (стандартное отклонение — $32$ секунды).
- Конверсия из заказа в назначение водителя на заказ равна $94\%$.
- Среднее время подачи такси после назначения водителя на заказ составляет $5$ минут и $16$ секунд (стандартное отклонение — $1$ минута и $5$ секунд).
- Конверсия из назначения в прибытие автомобиля в точку А составляет $82\%$.
- Среднее время поездки (от точки А до точки В) составляет $24$ минуты и $36$ секунд (стандартное отклонение — $7$ минут и $2$ секунды).
- Конверсия из прибытия автомобиля в точку А в прибытие в точку В составляет $88\%$.

### Задача 1. Автоматизация доверительного интервала (3 балла)

Создай функцию, которая будет проводить автоматизированный расчёт границ доверительного интервала для метрики.

Функция берёт на вход:
- Среднее значение метрики
- Стандартное отклонение метрики
- Количество наблюдений
- Уровень значимости $\alpha$ со значением по умолчанию $5\%$

Функция выдаёт:
- Левую границу доверительного интервала
- Правую границу доверительного интервала

Примени функцию к набору метрик:

- Конверсия из заказа в назначение водителя на заказ.
- Конверсия из назначения в прибытие водителя в точку А.
- Конверсия из прибытия в точку А в прибытие в точку В.
- Время между созданием заказа и назначением водителя на заказ.
- Время между назначением водителя на заказ и прибытием в точку А.
- Время между прибытием в точку А и прибытием в точку В.

*Квантиль нормального стандартного распределения необходимо рассчитать внутри функции.*

In [10]:
### Твой код здесь
from scipy.stats import norm

def get_z_critical(alpha):
    return norm.ppf(1 - alpha / 2)

def make_CI(mean, s, n, alpha=0.05):
  return mean - get_z_critical(alpha) * (s/np.sqrt(n)), mean + get_z_critical(alpha) * (s/np.sqrt(n))

In [2]:
n_orders = 8770

n_assigned = round(n_orders * 0.94)
n_arrived_A = round(n_assigned * 0.82)
n_arrived_B = round(n_arrived_A * 0.88)

In [11]:
p1 = 0.94
std1 = np.sqrt(p1 * (1 - p1))

CI1 = make_CI(p1, std1, n_orders)
CI1

(np.float64(0.9350296399910236), np.float64(0.9449703600089763))

In [12]:
p2 = 0.82
std2 = np.sqrt(p2 * (1 - p2))

CI2 = make_CI(p2, std2, n_assigned)
CI2

(np.float64(0.8117067973419891), np.float64(0.8282932026580108))

In [13]:
p3 = 0.88
std3 = np.sqrt(p3 * (1 - p3))

CI3 = make_CI(p3, std3, n_arrived_A)
CI3

(np.float64(0.8722534790070079), np.float64(0.8877465209929921))

In [21]:
t1 = 121
s1 = 32
ci_time1 = make_CI(t1, s1, n_assigned)
ci_time1

(np.float64(120.30923698282339), np.float64(121.69076301717661))

In [22]:
t2 = 316
s2 = 65
ci_time2 = make_CI(t2, s2, n_arrived_A)
ci_time2

(np.float64(314.4505124192386), np.float64(317.5494875807614))

In [23]:
t3 = 1476
s3 = 422
ci_time3 = make_CI(t3, s3, n_arrived_B)
ci_time3

(np.float64(1465.2764505818361), np.float64(1486.7235494181639))

## Задача 2. Среднее время до точки В (2 балла)

Можно ли утверждать, что среднее время от создания заказа до прибытия в точку **В** составляет не более $35$ минут?

<br>

**Подсказка**

Для ответа на вопрос рассчитайте границы доверительного интервала для суммы трёх показателей времени.

Для этого:
- Рассчитайте среднее суммы трёх показателей времени.
- Рассчитайте стандартную ошибку суммы трёх средних.
- Определите количество наблюдений для каждого показателя.
- Постройте доверительный интервал.

В качестве уровня значимости возьмите $5\%$.

In [25]:
### Твой код здесь
mean_time = t1 + t2 + t3

std_time = np.sqrt((s1**2 / n_assigned) + (s2**2 / n_arrived_A) + (s3**2 / n_arrived_B))

print((mean_time - get_z_critical(0.05) * std_time) / 60, (mean_time + get_z_critical(0.05) * std_time) / 60)

31.70238476986798 32.06428189679869


C вероятностью доверия 95% среднее суммарное время находится примерно от 31.7 до 32.1 минуты. Верхняя граница значительно ниже 35 минут, поэтому утверждение поддерживается данными.

## Задача 3. Ожидаемая выручка (2 балла)

Предположите, что все заказы были равномерно распределены по $10$ дням, а конверсия была неизменной в течение всего периода времени.

Средняя стоимость одного заказа составляет $750$ рублей.

Какую максимальную ожидаемую выручку за один среднестатистический день можно получить, если использовать правую границу $95\%$ доверительного интервала для конверсии из заказа в завершённую поездку?

В качестве уровня значимости возьмите $5\%$.

Округлите ответ до $0$ знаков после запятой.

<br>

*Подсказка*

Рассчитайте границы доверительного интервала для конверсии из заказа в завершённую поездку, которая является произведением трёх звеньев конверсии.

In [29]:
### Твой код здесь
p = p1 * p2 * p3
std = np.sqrt(p * (1 - p))

ci = make_CI(p, std, n_orders)

one_day_orders = n_orders / 10

max_day_orders = one_day_orders * ci[1]

one_day_money = max_day_orders * 750

round(one_day_money)

452585

## Задача 4. Бутстрап для времени поездки (3 балла)

Предположим, что время поездки подчиняется логнормальному распределению.

Это типично для показателей, которые не могут быть отрицательными и имеют длинный правый "хвост" — небольшое количество очень долгих поездок.

Параметры распределения:

- $\mu = 3.0$
- $\sigma = 0.8$

**Задание**

1. Сгенерируйте исходную выборку размера $n = 1000$ из логнормального распределения с заданными параметрами. Это будет наша "реальная" выборка времени поездок в минутах.

2. Реализуйте бутстрап-процедуру:

   - Сгенерируйте $B = 5000$ бутстрап-выборок — повторных выборок с возвращением — того же размера $n$ из исходной выборки.
   - Для каждой бутстрап-выборки вычислите целевые статистики:
     - Среднее;
     - Медиану;
     - Дисперсию.

In [33]:
### Твой код здесь
np.random.seed(42)

mu = 3.0
sigma = 0.8
n = 1000
B = 5000

sample = np.random.lognormal(
    mean=mu,
    sigma=sigma,
    size=n
)

bootstrap_means = np.empty(B)
bootstrap_medians = np.empty(B)
bootstrap_vars = np.empty(B)

for i in range(B):
    bootstrap_sample = np.random.choice(
        sample,
        size=n,
        replace=True
    )

    bootstrap_means[i] = np.mean(bootstrap_sample)
    bootstrap_medians[i] = np.median(bootstrap_sample)
    bootstrap_vars[i] = np.var(bootstrap_sample, ddof=1)

print("Исходная выборка:")
print("Среднее:", np.mean(sample))
print("Медиана:", np.median(sample))
print("Дисперсия:", np.var(sample, ddof=1))

print("\nБутстрап-распределения:")
print("Среднее бутстрап-средних:", np.mean(bootstrap_means))
print("Среднее бутстрап-медиан:", np.mean(bootstrap_medians))
print("Среднее бутстрап-дисперсий:", np.mean(bootstrap_vars))

Исходная выборка:
Среднее: 28.014155585734073
Медиана: 20.496224321073203
Дисперсия: 790.151948351544

Бутстрап-распределения:
Среднее бутстрап-средних: 28.01406751969776
Среднее бутстрап-медиан: 20.480406715162122
Среднее бутстрап-дисперсий: 788.1553429876551
